# GPU test for FaceFusion (Colab)

Confirms this runtime actually has a working CUDA GPU **before** you run the FaceFusion notebook.

First: **Runtime → Change runtime type → T4 GPU → Save**, then **Runtime → Disconnect and delete runtime** and **Reconnect**. Selecting GPU does nothing until the runtime restarts.

Then run the cells top to bottom and read the verdict in the last cell.

## 1. Is a physical GPU attached? (the definitive check)
A table with a GPU (e.g. *Tesla T4*) = good. `command not found` = no GPU on this machine.

In [ ]:
!nvidia-smi

: 

: 

## 2. onnxruntime-gpu install + advertised providers
Matches the version the fork's `install.py` uses. NOTE: a provider appearing here only means the *package* supports it — not that a GPU is present.

In [ ]:
!pip install -q onnxruntime-gpu==1.24.4
import onnxruntime as ort
print("onnxruntime:", ort.__version__)
print("build device:", ort.get_device())
print("available providers:", ort.get_available_providers())

: 

## 3. Can onnxruntime actually run on CUDA? (real runtime test)
Builds a tiny model and forces a CUDA session. If CUDA can't initialize, onnxruntime silently drops it and the active providers fall back to CPU only.

In [ ]:
!pip install -q onnx
import numpy as np
import onnx
from onnx import helper, TensorProto
import onnxruntime as ort

x = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1])
y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1])
node = helper.make_node('Identity', ['X'], ['Y'])
graph = helper.make_graph([node], 'g', [x], [y])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 13)])
onnx.save(model, 'identity.onnx')

sess = ort.InferenceSession('identity.onnx', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
active = sess.get_providers()
out = sess.run(None, {'X': np.array([1.0], dtype=np.float32)})
print("active providers:", active)
print("inference ok:", out)

## 4. Verdict

In [ ]:
import shutil

has_smi = shutil.which('nvidia-smi') is not None
cuda_active = 'CUDAExecutionProvider' in active

if has_smi and cuda_active:
    print('\u2705 GPU is working. FaceFusion will run on CUDA.')
elif not has_smi:
    print('\u274c No GPU attached to this runtime (nvidia-smi missing).')
    print('   Fix: Runtime -> Change runtime type -> T4 GPU -> Save,')
    print('        then Runtime -> Disconnect and delete runtime -> Reconnect.')
else:
    print('\u26a0\ufe0f  GPU present but onnxruntime fell back to CPU (CUDA/cuDNN mismatch).')
    print('   Try: !pip install -q onnxruntime-gpu==1.23.2 --force-reinstall, then rerun.')

: 